In [1]:
# Business Entity Resolution — Fresh End-to-End Winning Pipeline
- **Stage 1**: Train Triple-GBDT Ensemble (LightGBM + XGBoost + CatBoost) on 400,000 Mined Pairs
- **Stage 2**: Evaluate Validation Macro F_0.5 and Apply Tau = 0.72 Precision Guard
- **Stage 3**: High-Speed Streaming Test Inference across France (Zero-Shot), US, and India
- **Stage 4**: Format Validation & Creation of submission_final.zip


SyntaxError: invalid syntax (154940344.py, line 2)

In [2]:
!pip install -q rapidfuzz lightgbm xgboost catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.6 MB/s eta 0:00:00:00:010:01


In [3]:
# -*- coding: utf-8 -*-
"""
ML Challenge 2026: Business Entity Resolution - Gold Standard Pipeline (Target: 0.985+ Macro F_0.5).
Triple-GBDT Ensemble (LightGBM + XGBoost + CatBoost) with:
1. True Hard-Negative Mining from the full candidate pool
2. Address Number Conflict Veto & Postal Conflict Veto
3. Branch Modifier Conflict Detection
4. Real Out-of-Fold Validation & Threshold Calibration
5. 1-to-2 Match-Per-Source Precision Filtering
"""

import collections
import gc
import glob
import os
import pickle
import re
import sys
import time
import unicodedata
import warnings
import zipfile
from typing import Dict, List, Set, Tuple, Union, Any

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from rapidfuzz import fuzz
from rapidfuzz.distance import Levenshtein, JaroWinkler

warnings.filterwarnings("ignore")

# ==============================================================================
# 1. AUTO-DETECT DATASET PATHS (TRAIN & TEST)
# ==============================================================================
print("="*80)
print("STAGE 0: AUTO-DETECTING KAGGLE INPUT PATHS")
print("="*80)

def find_file(filename: str) -> str:
    patterns = [f"/kaggle/input/**/{filename}", f"./**/{filename}", f"../**/{filename}"]
    for pat in patterns:
        matches = glob.glob(pat, recursive=True)
        if matches:
            return matches[0]
    return ""

test_s1_path = find_file("test_source1.tsv")
train_s1_path = find_file("train_source1.tsv")
gt_path = find_file("train_ground_truth.tsv")

if not test_s1_path:
    TEST_DIR = r"c:\Users\hetha\Desktop\ml_challenge\dataset\student_resource\dataset\test"
    TRAIN_DIR = r"c:\Users\hetha\Desktop\ml_challenge\dataset\student_resource\dataset\train"
else:
    TEST_DIR = os.path.dirname(test_s1_path)
    TRAIN_DIR = os.path.dirname(train_s1_path)

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"  -> Train Directory: {TRAIN_DIR}")
print(f"  -> Test Directory:  {TEST_DIR}")
print(f"  -> Output Directory: {OUTPUT_DIR}")


# ==============================================================================
# 2. ADVANCED PREPROCESSING & FEATURE DEFINITIONS
# ==============================================================================
RE_TOKEN_SPLIT = re.compile(r'[^\w]+', re.UNICODE)
RE_NUM_EXTRACT = re.compile(r'\b\d{1,6}\b')
RE_POSTAL_EXTRACT = re.compile(r'\b\d{5,6}\b')

NAME_STOP_WORDS = {
    'the', 'and', 'for', 'with', 'from', 'near', 'pvt', 'ltd', 'inc', 'corp',
    'llc', 'llp', 'sarl', 'sas', 'eurl', 'sci', 'sa', 'snc', 'private', 'limited',
    'corporation', 'incorporated', 'company', 'co', 'store', 'shop', 'services',
    'group', 'center', 'centre', 'enterprises', 'solutions', 'international', 'of',
    'de', 'la', 'le', 'et', 'du', 'des', 'en', 'a', 'au', 'aux', 'd', 'l'
}

ADDR_STOP_WORDS = {
    'road', 'street', 'st', 'rd', 'ave', 'avenue', 'dr', 'drive', 'blvd',
    'boulevard', 'lane', 'ln', 'court', 'ct', 'highway', 'hwy', 'box', 'po',
    'floor', 'fl', 'suite', 'ste', 'apt', 'apartment', 'bldg', 'building',
    'near', 'opposite', 'behind', 'plot', 'house', 'no', 'hno', 'flat', 'unit',
    'rue', 'allee', 'cours', 'place', 'pl', 'impasse', 'chemin', 'cedex'
}

BRANCH_MODIFIERS = {
    'central', 'west', 'east', 'north', 'south', 'plaza', 'mall', 'services',
    'holdings', 'division', 'branch', 'station', 'subway', 'airport', 'hospital',
    'clinic', 'express', 'bazaar', 'supermarket', 'mart'
}

def remove_diacritics(text: str) -> str:
    if not text:
        return ""
    return "".join(c for c in unicodedata.normalize('NFKD', text) if not unicodedata.combining(c))

def parse_record_fast(rec: Dict[str, str]) -> Dict[str, Any]:
    name_clean = remove_diacritics(str(rec.get('business_name', ''))).lower()
    addr_clean = remove_diacritics(str(rec.get('business_address', ''))).lower()
    
    tokens = [t for t in RE_TOKEN_SPLIT.split(name_clean) if len(t) >= 2]
    sig_w = [t for t in tokens if t not in NAME_STOP_WORDS]
    
    raw_nums = RE_NUM_EXTRACT.findall(addr_clean)
    nums = {n.lstrip('0') or '0' for n in raw_nums}
    posts = set(RE_POSTAL_EXTRACT.findall(addr_clean))
    
    branches = set(tokens) & BRANCH_MODIFIERS
    
    return {
        'entity_id': rec.get('entity_id', ''),
        'name': name_clean,
        'addr': addr_clean,
        'tokens': set(tokens),
        'sig_w': sig_w,
        'nums': nums,
        'posts': posts,
        'branches': branches
    }

def extract_fast_blocking_keys(name: str, address: str) -> Set[str]:
    keys = set()
    name_clean = remove_diacritics(str(name)).lower() if name else ""
    tokens = [t for t in RE_TOKEN_SPLIT.split(name_clean) if t]
    sig_words = [t for t in tokens if len(t) >= 2 and t not in NAME_STOP_WORDS]
    
    for w in sig_words[:6]:
        keys.add(f"w_{w}")
    for i in range(len(sig_words) - 1):
        keys.add(f"sh_{sig_words[i]}_{sig_words[i+1]}")
    if len(sig_words) >= 2:
        ac = "".join(w[0] for w in sig_words[:5])
        if len(ac) >= 2:
            keys.add(f"ac_{ac}")
            
    addr_clean = remove_diacritics(str(address)).lower() if address else ""
    addr_tokens = [t for t in RE_TOKEN_SPLIT.split(addr_clean) if t]
    addr_sig = [t for t in addr_tokens if len(t) >= 3 and t not in ADDR_STOP_WORDS and not t.isdigit()]
    for aw in addr_sig[:5]:
        keys.add(f"aw_{aw}")
    for i in range(len(addr_sig) - 1):
        keys.add(f"ash_{addr_sig[i]}_{addr_sig[i+1]}")
        
    postals = RE_POSTAL_EXTRACT.findall(addr_clean)
    for p in postals:
        keys.add(f"post_{p}")
        
    raw_nums = RE_NUM_EXTRACT.findall(addr_clean)
    nums = [n.lstrip('0') or '0' for n in raw_nums]
    first_name_word = sig_words[0] if sig_words else (tokens[0] if tokens else "")
    for n in nums[:2]:
        keys.add(f"num_{n}")
        if first_name_word:
            keys.add(f"numw_{n}_{first_name_word}")
            
    return keys

FEATURE_NAMES = [
    'name_ratio', 'name_sort', 'name_set', 'name_jw', 'len_diff', 'tok_jacc',
    'addr_ratio', 'addr_sort', 'addr_set', 'addr_jw',
    'num_match', 'num_conflict', 'num_overlap',
    'post_match', 'post_conflict', 'branch_conflict', 'harmonic',
    'is_s2', 'is_s3'
]

def compute_pair_features(s1: Dict[str, Any], cand: Dict[str, Any]) -> List[float]:
    n_ratio = fuzz.ratio(s1['name'], cand['name']) / 100.0
    n_sort = fuzz.token_sort_ratio(s1['name'], cand['name']) / 100.0
    n_set = fuzz.token_set_ratio(s1['name'], cand['name']) / 100.0
    n_jw = JaroWinkler.similarity(s1['name'], cand['name'])
    
    len_diff = abs(len(s1['name']) - len(cand['name'])) / max(len(s1['name']), len(cand['name']), 1)
    tok_inter = len(s1['tokens'] & cand['tokens'])
    tok_union = len(s1['tokens'] | cand['tokens'])
    tok_jacc = tok_inter / max(tok_union, 1)
    
    a_ratio = fuzz.ratio(s1['addr'], cand['addr']) / 100.0
    a_sort = fuzz.token_sort_ratio(s1['addr'], cand['addr']) / 100.0
    a_set = fuzz.token_set_ratio(s1['addr'], cand['addr']) / 100.0
    a_jw = JaroWinkler.similarity(s1['addr'], cand['addr'])
    
    num_match = 1.0 if (s1['nums'] and cand['nums'] and (s1['nums'] & cand['nums'])) else 0.0
    num_conflict = 1.0 if (s1['nums'] and cand['nums'] and not (s1['nums'] & cand['nums'])) else 0.0
    num_overlap = len(s1['nums'] & cand['nums']) / max(len(s1['nums'] | cand['nums']), 1) if (s1['nums'] and cand['nums']) else 0.0
    
    post_match = 1.0 if (s1['posts'] and cand['posts'] and (s1['posts'] & cand['posts'])) else 0.0
    post_conflict = 1.0 if (s1['posts'] and cand['posts'] and not (s1['posts'] & cand['posts'])) else 0.0
    
    branch_conflict = 1.0 if (s1['branches'] != cand['branches']) else 0.0
    harmonic = (2.0 * n_set * a_set) / (n_set + a_set + 1e-5)
    
    is_s2 = 1.0 if cand['entity_id'].startswith('S2-') else 0.0
    is_s3 = 1.0 if cand['entity_id'].startswith('S3-') else 0.0
    
    return [
        n_ratio, n_sort, n_set, n_jw, len_diff, tok_jacc,
        a_ratio, a_sort, a_set, a_jw,
        num_match, num_conflict, num_overlap,
        post_match, post_conflict, branch_conflict, harmonic,
        is_s2, is_s3
    ]


# ==============================================================================
# 3. METRIC DEFINITION
# ==============================================================================
def compute_entity_f05(true_ids: Set[str], pred_ids: Set[str]) -> float:
    if not true_ids:
        return 1.0 if not pred_ids else 0.0
    if not pred_ids:
        return 0.0
    tp = len(true_ids & pred_ids)
    if tp == 0:
        return 0.0
    precision = tp / len(pred_ids)
    recall = tp / len(true_ids)
    denom = 0.25 * precision + recall
    return (1.25 * precision * recall) / denom if denom > 0 else 0.0

def compute_macro_f05(ground_truth: Dict[str, Set[str]], predictions: Dict[str, Set[str]]) -> Dict[str, float]:
    scores = [compute_entity_f05(true_ids, predictions.get(s1_id, set())) for s1_id, true_ids in ground_truth.items()]
    return {'macro_f05': float(np.mean(scores)) if scores else 0.0}


# ==============================================================================
# 4. OPTIMIZED INVERTED INDEX BLOCKING
# ==============================================================================
class OptimizedInvertedIndex:
    def __init__(self, max_posting_size: int = 2000):
        self.index = collections.defaultdict(list)
        self.max_posting_size = max_posting_size

    def add_doc(self, doc_id: str, keys: Set[str]):
        for k in keys:
            posting = self.index[k]
            if len(posting) < self.max_posting_size:
                posting.append(doc_id)

    def query(self, keys: Set[str], top_k: int = 25) -> List[Tuple[str, int]]:
        counts = collections.Counter()
        for k in keys:
            posting = self.index.get(k)
            if posting and len(posting) <= 500:
                if k.startswith(('sh_', 'ash_', 'numw_', 'post_')):
                    weight = 5
                elif k.startswith(('ac_', 'w_')):
                    weight = 3
                elif k.startswith('aw_'):
                    weight = 2
                else:
                    weight = 1
                for doc_id in posting:
                    counts[doc_id] += weight
        return counts.most_common(top_k)


# ==============================================================================
# 5. TRIPLE-GBDT ENSEMBLE
# ==============================================================================
class TripleGBDTEnsemble:
    def __init__(self):
        self.lgb_model = lgb.LGBMClassifier(
            n_estimators=300,
            learning_rate=0.06,
            num_leaves=63,
            max_depth=8,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=42,
            n_jobs=-1
        )
        self.xgb_model = xgb.XGBClassifier(
            n_estimators=300,
            learning_rate=0.06,
            max_depth=6,
            subsample=0.85,
            colsample_bytree=0.85,
            random_state=42,
            n_jobs=-1,
            tree_method='hist'
        )
        self.cb_model = cb.CatBoostClassifier(
            iterations=300,
            learning_rate=0.06,
            depth=6,
            random_seed=42,
            verbose=False,
            thread_count=-1
        )
        
    def fit(self, X: np.ndarray, y: np.ndarray):
        print(f"Fitting Triple-GBDT Ensemble on {len(X):,} candidate pairs...", flush=True)
        t0 = time.time()
        self.lgb_model.fit(X, y)
        print(f"  -> LightGBM trained ({time.time() - t0:.1f}s)")
        t0 = time.time()
        self.xgb_model.fit(X, y)
        print(f"  -> XGBoost trained ({time.time() - t0:.1f}s)")
        t0 = time.time()
        self.cb_model.fit(X, y)
        print(f"  -> CatBoost trained ({time.time() - t0:.1f}s)")
        
    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        p_lgb = self.lgb_model.predict_proba(X)[:, 1]
        p_xgb = self.xgb_model.predict_proba(X)[:, 1]
        p_cb = self.cb_model.predict_proba(X)[:, 1]
        return 0.40 * p_lgb + 0.35 * p_xgb + 0.25 * p_cb


# ==============================================================================
# 6. HARD-NEGATIVE TRAINING & OUT-OF-FOLD CALIBRATION
# ==============================================================================
def train_and_calibrate_pipeline() -> Tuple[TripleGBDTEnsemble, float]:
    print("\n" + "="*80)
    print("STAGE 1: HARD-NEGATIVE MINING, TRAINING & OUT-OF-FOLD CALIBRATION")
    print("="*80)
    
    t_start = time.time()
    
    # 1. Load Training S1 and Ground Truth
    print("Loading 45,000 S1 training entities and ground truth...", flush=True)
    df_s1 = pd.read_csv(os.path.join(TRAIN_DIR, "train_source1.tsv"), sep="\t", nrows=45000).fillna('')
    s1_all_ids = set(df_s1['entity_id'])
    
    df_gt = pd.read_csv(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"), sep="\t")
    df_gt = df_gt[df_gt['source1_entity_id'].isin(s1_all_ids)]
    gt_map = {}
    all_targets = set()
    for _, row in df_gt.iterrows():
        s1 = str(row['source1_entity_id'])
        m_str = str(row['matched_entity_ids']) if pd.notna(row['matched_entity_ids']) else ""
        targs = {x.strip() for x in m_str.split(',') if x.strip()}
        gt_map[s1] = targs
        all_targets.update(targs)
        
    print(f"Loaded {len(s1_all_ids):,} S1 queries ({len(all_targets):,} true targets in GT).")
    
    # 2. Load Target Pool (True Targets + Distractor Pool)
    print("Loading candidate target pool from Source 2 & Source 3...", flush=True)
    df_s2_dist = pd.read_csv(os.path.join(TRAIN_DIR, "train_source2.tsv"), sep="\t", nrows=100000).fillna('')
    df_s3_dist = pd.read_csv(os.path.join(TRAIN_DIR, "train_source3.tsv"), sep="\t", nrows=100000).fillna('')
    df_s2_full = pd.read_csv(os.path.join(TRAIN_DIR, "train_source2.tsv"), sep="\t").fillna('')
    df_s3_full = pd.read_csv(os.path.join(TRAIN_DIR, "train_source3.tsv"), sep="\t").fillna('')
    
    df_s2_targ = df_s2_full[df_s2_full['entity_id'].isin(all_targets)]
    df_s3_targ = df_s3_full[df_s3_full['entity_id'].isin(all_targets)]
    
    pool_df = pd.concat([df_s2_dist, df_s3_dist, df_s2_targ, df_s3_targ], ignore_index=True).drop_duplicates('entity_id')
    del df_s2_dist, df_s3_dist, df_s2_full, df_s3_full, df_s2_targ, df_s3_targ
    gc.collect()
    print(f"Target pool contains {len(pool_df):,} records.")
    
    s1_parsed = {r['entity_id']: parse_record_fast(r) for r in df_s1.to_dict('records')}
    pool_parsed = {r['entity_id']: parse_record_fast(r) for r in pool_df.to_dict('records')}
    del pool_df
    gc.collect()
    
    # 3. Build Inverted Index on Pool
    print("Building Inverted Index on Target Pool...", flush=True)
    inv_index = OptimizedInvertedIndex(max_posting_size=2000)
    for cid, cand in pool_parsed.items():
        keys = extract_fast_blocking_keys(cand['name'], cand['addr'])
        inv_index.add_doc(cid, keys)
        
    # 4. Partition into Train (35,000) and Strictly Held-Out Validation (10,000)
    s1_id_list = list(df_s1['entity_id'])
    train_ids = s1_id_list[:35000]
    val_ids = s1_id_list[35000:]
    
    # 5. Mine Training Pairs (True Positives + Real Confusable Hard Negatives)
    print(f"Mining candidate pairs for {len(train_ids):,} training queries...", flush=True)
    X_train, y_train = [], []
    for s1_id in train_ids:
        s1 = s1_parsed[s1_id]
        true_targs = gt_map.get(s1_id, set())
        keys = extract_fast_blocking_keys(s1['name'], s1['addr'])
        top_matches = inv_index.query(keys, top_k=25)
        cands = {cid for cid, _ in top_matches} | true_targs
        
        for cid in cands:
            cand = pool_parsed.get(cid)
            if cand:
                feats = compute_pair_features(s1, cand)
                label = 1.0 if cid in true_targs else 0.0
                X_train.append(feats)
                y_train.append(label)
                
    X_train = np.array(X_train, dtype=np.float32)
    y_train = np.array(y_train, dtype=np.float32)
    print(f"Training Feature Matrix: {X_train.shape}, True Positives: {int(y_train.sum()):,}, Hard Negatives: {len(y_train) - int(y_train.sum()):,}")
    
    # 6. Fit Triple-GBDT Ensemble
    ensemble = TripleGBDTEnsemble()
    ensemble.fit(X_train, y_train)
    del X_train, y_train
    gc.collect()
    
    # 7. Real Out-of-Fold Validation on Held-Out 10,000 Queries
    print(f"\nRunning Honest Out-of-Fold Validation on {len(val_ids):,} held-out queries...", flush=True)
    val_pairs = []
    val_tracking = []
    for s1_id in val_ids:
        s1 = s1_parsed[s1_id]
        keys = extract_fast_blocking_keys(s1['name'], s1['addr'])
        top_matches = inv_index.query(keys, top_k=25)
        for cid, _ in top_matches:
            cand = pool_parsed.get(cid)
            if cand:
                val_pairs.append(compute_pair_features(s1, cand))
                val_tracking.append((s1_id, cid))
                
    X_val = np.array(val_pairs, dtype=np.float32)
    val_probs = ensemble.predict_proba(X_val)
    
    print("\n" + "-"*60)
    print("CALIBRATING PRECISION THRESHOLD ON OUT-OF-FOLD SPLIT")
    print("-"*60)
    
    best_tau = 0.70
    best_macro = 0.0
    val_gt = {s: gt_map.get(s, set()) for s in val_ids}
    
    # Sweep thresholds and apply Veto Constraints
    for tau in [0.55, 0.60, 0.65, 0.70, 0.72, 0.75, 0.78, 0.80, 0.85]:
        preds = collections.defaultdict(list)
        for idx, (s1_id, cid) in enumerate(val_tracking):
            # 1. Address Number Conflict Veto: Index 11 is num_conflict
            if X_val[idx, 11] == 1.0:
                continue
            # 2. Postal Code Conflict Veto: Index 14 is post_conflict
            if X_val[idx, 14] == 1.0:
                continue
            # 3. Branch Conflict Veto: Index 15
            if X_val[idx, 15] == 1.0 and X_val[idx, 2] < 0.95:
                continue
            # 4. Probability Threshold
            prob = val_probs[idx]
            if prob >= tau:
                preds[s1_id].append((cid, prob))
                
        # Apply Top-2 Matches-Per-Source Cap
        final_preds = {}
        for s1_id in val_ids:
            cand_list = preds.get(s1_id, [])
            if not cand_list:
                final_preds[s1_id] = set()
            else:
                s2_cands = sorted([c for c, p in cand_list if c.startswith('S2-')], key=lambda x: [p for c, p in cand_list if c == x][0], reverse=True)[:2]
                s3_cands = sorted([c for c, p in cand_list if c.startswith('S3-')], key=lambda x: [p for c, p in cand_list if c == x][0], reverse=True)[:2]
                final_preds[s1_id] = set(s2_cands + s3_cands)
                
        res = compute_macro_f05(val_gt, final_preds)
        score = res['macro_f05']
        print(f"  Threshold tau = {tau:.2f} -> Out-of-Fold Macro F_0.5: {score:.4f} ({score*100:.2f}%)")
        if score > best_macro:
            best_macro = score
            best_tau = tau
            
    print(f"\n>>> OPTIMAL CALIBRATED THRESHOLD: tau = {best_tau:.2f} (Macro F_0.5 = {best_macro*100:.2f}%) <<<")
    print(f"Training & Validation completed in {time.time() - t_start:.1f}s.")
    
    del df_s1, s1_parsed, pool_parsed, inv_index, X_val, val_probs, val_pairs, val_tracking
    gc.collect()
    
    return ensemble, best_tau


# ==============================================================================
# 7. COUNTRY-PARTITIONED STREAMING TEST INFERENCE WITH PRECISION GATES
# ==============================================================================
def load_country_records(filepath: str, target_country: str) -> List[Dict[str, str]]:
    records = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        header_line = f.readline().rstrip('\r\n')
        cols = [c.strip().lower() for c in header_line.split('\t')]
        id_idx = cols.index('entity_id')
        name_idx = cols.index('business_name')
        addr_idx = cols.index('business_address')
        c_idx = cols.index('country')
        for line in f:
            parts = line.rstrip('\r\n').split('\t')
            if len(parts) > c_idx and parts[c_idx].strip() == target_country:
                records.append({
                    'entity_id': parts[id_idx].strip(),
                    'business_name': parts[name_idx].strip() if len(parts) > name_idx else '',
                    'business_address': parts[addr_idx].strip() if len(parts) > addr_idx else '',
                    'country': target_country
                })
    return records


def run_full_test_inference(ensemble: TripleGBDTEnsemble, optimal_tau: float):
    t_start = time.time()
    print("\n" + "="*80)
    print(f"STAGE 2: STREAMING TEST INFERENCE AT OPTIMAL TAU = {optimal_tau:.2f}")
    print("="*80)
    
    matching_file = os.path.join(OUTPUT_DIR, "matching_results.tsv")
    candidate_file = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
    
    print("Reading full test_source1.tsv entity sequence...", flush=True)
    s1_all_ids = []
    with open(os.path.join(TEST_DIR, "test_source1.tsv"), 'r', encoding='utf-8', errors='ignore') as f:
        f.readline()
        for line in f:
            parts = line.rstrip('\r\n').split('\t')
            if parts[0].strip():
                s1_all_ids.append(parts[0].strip())
    print(f"Total Source 1 test entities: {len(s1_all_ids):,}", flush=True)
    
    all_matching_dict = {}
    all_candidate_dict = {}
    
    chunk_size = 35000
    countries = ['France', 'US', 'India']
    
    for country in countries:
        t_country = time.time()
        print(f"\n" + "-"*70)
        print(f"PROCESSING COUNTRY: {country}")
        print(f"-"*70)
        
        s1_country = load_country_records(os.path.join(TEST_DIR, "test_source1.tsv"), country)
        s2_country = load_country_records(os.path.join(TEST_DIR, "test_source2.tsv"), country)
        s3_country = load_country_records(os.path.join(TEST_DIR, "test_source3.tsv"), country)
        s23_country = s2_country + s3_country
        del s2_country, s3_country
        gc.collect()
        
        print(f"  Loaded {len(s1_country):,} S1 queries, {len(s23_country):,} target records.")
        
        # Build Index
        t_block = time.time()
        inv_index = OptimizedInvertedIndex(max_posting_size=2000)
        for rec in s23_country:
            keys = extract_fast_blocking_keys(rec.get('business_name', ''), rec.get('business_address', ''))
            inv_index.add_doc(rec['entity_id'], keys)
            
        cands_map = {}
        for rec in s1_country:
            s1_id = rec['entity_id']
            keys = extract_fast_blocking_keys(rec.get('business_name', ''), rec.get('business_address', ''))
            top_matches = inv_index.query(keys, top_k=25)
            cands_map[s1_id] = [doc_id for doc_id, _ in top_matches]
            
        print(f"  Candidate generation completed in {time.time() - t_block:.1f}s")
        
        needed_cands = {cid for clist in cands_map.values() for cid in clist}
        s23_parsed_cache = {}
        for rec in s23_country:
            if rec['entity_id'] in needed_cands:
                s23_parsed_cache[rec['entity_id']] = parse_record_fast(rec)
        del s23_country, inv_index
        gc.collect()
        
        total_batches = (len(s1_country) + chunk_size - 1) // chunk_size
        print(f"  Scoring {len(s1_country):,} queries across {total_batches} chunks with precision gates...")
        
        country_matches = 0
        country_pairs = 0
        
        for batch_idx, batch_start in enumerate(range(0, len(s1_country), chunk_size), 1):
            t_batch = time.time()
            batch_s1 = s1_country[batch_start:batch_start + chunk_size]
            batch_pairs = []
            pair_ptrs = []
            
            for s1_rec in batch_s1:
                s1_id = s1_rec['entity_id']
                cand_list = cands_map.get(s1_id, [])
                all_candidate_dict[s1_id] = cand_list
                
                if not cand_list:
                    all_matching_dict[s1_id] = []
                    continue
                    
                s1_p = parse_record_fast(s1_rec)
                for cid in cand_list:
                    c_p = s23_parsed_cache.get(cid)
                    if c_p:
                        batch_pairs.append((s1_p, c_p))
                        pair_ptrs.append((s1_id, cid))
                        
            batch_matches = 0
            if batch_pairs:
                feat_matrix = np.array([compute_pair_features(s1_p, c_p) for s1_p, c_p in batch_pairs], dtype=np.float32)
                probs = ensemble.predict_proba(feat_matrix)
                
                # Group predicted candidates per S1 entity
                entity_cands = collections.defaultdict(list)
                for idx, (s1_id, cid) in enumerate(pair_ptrs):
                    # 1. Number Conflict Veto: Index 11
                    if feat_matrix[idx, 11] == 1.0:
                        continue
                    # 2. Postal Conflict Veto: Index 14
                    if feat_matrix[idx, 14] == 1.0:
                        continue
                    # 3. Branch Conflict Veto: Index 15
                    if feat_matrix[idx, 15] == 1.0 and feat_matrix[idx, 2] < 0.95:
                        continue
                    # 4. Probability Gate
                    prob = probs[idx]
                    if prob >= optimal_tau:
                        entity_cands[s1_id].append((cid, prob))
                        
                # 5. Cap matches to top-2 per source (S2 and S3)
                for s1_rec in batch_s1:
                    s1_id = s1_rec['entity_id']
                    cand_matches = entity_cands.get(s1_id, [])
                    if not cand_matches:
                        all_matching_dict[s1_id] = []
                    else:
                        s2_cands = sorted([c for c, p in cand_matches if c.startswith('S2-')], key=lambda x: [p for c, p in cand_matches if c == x][0], reverse=True)[:2]
                        s3_cands = sorted([c for c, p in cand_matches if c.startswith('S3-')], key=lambda x: [p for c, p in cand_matches if c == x][0], reverse=True)[:2]
                        selected = s2_cands + s3_cands
                        all_matching_dict[s1_id] = selected
                        batch_matches += len(selected)
            else:
                for s1_rec in batch_s1:
                    all_matching_dict[s1_rec['entity_id']] = []
                    
            country_matches += batch_matches
            country_pairs += len(batch_pairs)
            scored = min(batch_start + chunk_size, len(s1_country))
            pct = (scored / len(s1_country)) * 100.0
            print(f"    [{country}] Batch {batch_idx:02d}/{total_batches:02d} ({pct:5.1f}%) | Scored {scored:,}/{len(s1_country):,} | Pairs: {len(batch_pairs):,} | Matches: {batch_matches:,} | {time.time() - t_batch:.1f}s", flush=True)
            
        print(f"  {country} completed in {time.time() - t_country:.1f}s (Pairs: {country_pairs:,}, Matches: {country_matches:,})")
        del s1_country, s23_parsed_cache, cands_map
        gc.collect()
        
    # Write Final Output TSVs
    print("\nWriting final submission TSVs...", flush=True)
    with open(matching_file, 'w', encoding='utf-8') as f_match, \
         open(candidate_file, 'w', encoding='utf-8') as f_cand:
        
        f_match.write("source1_entity_id\tmatched_entity_ids\n")
        f_cand.write("source1_entity_id\tcandidate_entity_ids\n")
        
        for s1_id in s1_all_ids:
            matches = all_matching_dict.get(s1_id, [])
            cands = all_candidate_dict.get(s1_id, [])
            f_match.write(f"{s1_id}\t{','.join(matches)}\n")
            f_cand.write(f"{s1_id}\t{','.join(cands)}\n")
            
    print(f"Successfully generated all {len(s1_all_ids):,} rows in matching_results.tsv and candidate_pairs.tsv!")
    
    # Packaging
    zip_path = os.path.join(OUTPUT_DIR, "submission_gold.zip")
    print(f"Packaging {zip_path}...", flush=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(matching_file, arcname="output/matching_results.tsv")
        zf.write(candidate_file, arcname="output/candidate_pairs.tsv")
    print(f"Package created: {os.path.getsize(zip_path):,} bytes.")


# ==============================================================================
# MAIN ENTRY POINT
# ==============================================================================
if __name__ == '__main__':
    t_global = time.time()
    ensemble, optimal_tau = train_and_calibrate_pipeline()
    run_full_test_inference(ensemble, optimal_tau)
    print(f"\nALL STAGES COMPLETED SUCCESSFULLY IN {(time.time() - t_global)/60:.1f} MINUTES!")


STAGE 0: AUTO-DETECTING KAGGLE INPUT PATHS
  -> Train Directory: /kaggle/input/datasets/vedantdube123/dataset/student_resource/dataset/train
  -> Test Directory:  /kaggle/input/datasets/vedantdube123/dataset/student_resource/dataset/test
  -> Output Directory: /kaggle/working

STAGE 1: HARD-NEGATIVE MINING, TRAINING & OUT-OF-FOLD CALIBRATION
Loading 45,000 S1 training entities and ground truth...
Loaded 45,000 S1 queries (155,838 true targets in GT).
Loading candidate target pool from Source 2 & Source 3...
Target pool contains 352,861 records.
Building Inverted Index on Target Pool...
Mining candidate pairs for 35,000 training queries...
Training Feature Matrix: (879561, 19), True Positives: 121,105, Hard Negatives: 758,456
Fitting Triple-GBDT Ensemble on 879,561 candidate pairs...
[LightGBM] [Info] Number of positive: 121105, number of negative: 758456
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.057329 seconds.
You can set `force_row_wise=t

KeyboardInterrupt: 